# SGD regularizer grid on the tiny circuit

Thin interactive wrapper around **`sgd_grid.py`** — the experiment config
(task, shapes, wd/wn grids, LR tune) lives there, so this notebook and a
headless VM run (`uv run python sgd_grid.py all`) stay in sync.

The task: the low-data 8-wire circuit from `tiny_circuit.ipynb` (depth 4,
circuit seed 2, solo wire 0, `train_frac=0.5` — 128 of 256 inputs), trained
with SGD+momentum over **all combinations of `weight_decay` and
`weight_noise`** across the 7-shape model grid. `weight_noise` is
init-relative (fraction of each layer's init std). Stages: LR-tune (clean
runs) -> 63-run main grid -> big trajectory figure + summary. Everything is
idempotent and resumable.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import os
    %pip -q install -U "jax[cuda12]" optax
    if not os.path.exists("/content/circscale"):
        !git clone https://github.com/amdson/circscale.git /content/circscale
    %cd /content/circscale
    !git pull
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/circscale_runs", exist_ok=True)
    if not os.path.islink("runs"):
        os.symlink("/content/drive/MyDrive/circscale_runs", "runs")

## Setup

Everything comes from the module. To experiment with different grids here,
override before running the stages, e.g. `G.WN_GRID = [0.0, 0.1, 0.3]` —
globals are read at call time.

In [ ]:
import sgd_grid as G
from sgd_grid import (SHAPES, WD_GRID, WN_GRID, LR_GRID, STEPS, TARGET_WIRE,
                      n_params, stage_tune, stage_grid, grid_cfgs,
                      load_results, fig_grid, fig_summary)

print(f"task: {G.N_WIRES} wires, depth {G.CIRC_DEPTH}, seed {G.CIRCUIT_SEED}, "
      f"wire {TARGET_WIRE}, train_frac {G.TRAIN_FRAC}")
print(f"{len(SHAPES)} shapes x {len(WD_GRID)} wd x {len(WN_GRID)} wn "
      f"at {STEPS:,} steps -> {G.OUT_DIR}/")

## LR tune (clean runs; idempotent)

In [ ]:
tuned_lr = stage_tune()

## Main grid (idempotent — interrupt and re-run freely)

In [ ]:
cfgs = stage_grid(tuned_lr)

## The big figure

Rows: model shapes. Columns: (weight_decay, weight_noise) combinations.
Each panel: target-wire BCE on the training pool (dashed) vs held-out
inputs (solid), log-log; chance dotted.

In [ ]:
res = load_results(cfgs)
fig_grid(res);

## Summary — final held-out BCE / accuracy vs params

In [ ]:
fig_summary(res);

## Notes

- LR is tuned once per shape on the clean configuration and held fixed
  across the (wd, wn) grid — noisy runs might individually prefer a lower
  LR, so read strong-noise instability as "at the clean-tuned LR".
- `weight_noise` is the transient (ELBO-style) mode in init-relative units:
  each leaf gets std `wn x (its init std)`, so the function-space
  perturbation is comparable across widths (norm scales get no noise). Add
  `noise_mode="persist"` / `noise_scale="abs"` configs to compare.
- With SGD, `weight_decay` is classic L2-coupled decay.
- At `train_frac=0.5` on this task, exact recovery is in-principle
  possible (no consistent impostor circuits at 128 revealed inputs; the
  ambiguous regime starts around train_frac ~0.2), so held-out shortfalls
  are attributable to the network prior / optimization.
- Held-out accuracy has granularity 1/128. Momentum is 0.9 throughout.
- On a VM, skip this notebook entirely: `uv run python sgd_grid.py all`.